In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from pathlib import Path
from pprint import pprint
import sys
from typing import Optional, List, Dict, Any, Tuple
if '..' not in sys.path: sys.path.append('..')

import numpy as np
from pydantic_yaml import parse_yaml_file_as
import torch
from transformers import AutoTokenizer, PreTrainedTokenizer

from mllm.exp.args import MIXED_DECODER_MODEL_CFG_FNAME
from mllm.model.mixed_decoder import MixedDecoder
from mllm.config.model import MixedDecoderCfg
from mllm.data.qna.batch import QnaBatch
from mllm.data.qna.dataset import QnaDatasetAgg, QnaDatasetType, QNA_DATASETS_DEFAULT, QnaBaseDataset, load_qna_datasets

🚨 `emb_off` is part of GPT2Model.forward's signature, but not documented. Make sure to add it to the docstring of the function in q:\prog\mllm\notebooks\..\mllm\model\gpt2\modeling_gpt2.py.
🚨 `emb_off` is part of GPT2LMHeadModel.forward's signature, but not documented. Make sure to add it to the docstring of the function in q:\prog\mllm\notebooks\..\mllm\model\gpt2\modeling_gpt2.py.


## Config

In [23]:
DATA_PATH = Path('Q:/data')

TRAIN_ENCDEC_BERT_PATH = DATA_PATH / 'train_mllm_encdec_bert'
mixed_decoder_subdir = 'mixeddecoder-20260414_205525-pre_mixeddecoder20260316221645-bertbaseuncased-d768-embEncCls-inp128-decBertbaseuncased-msl384-sepT-pallF-eer4-ewn2x10-frzencF-dsQnaall-trn_lr5e-05_bs15'
# mixed_decoder_subdir = 'mixeddecoder-20260416_083543-pre_mixeddecoder20260304105309-bertbaseuncased-d768-embEncCls-inp128-decGpt2-msl384-sepT-pallF-eer4-ewn2x10-frzencF-dsQnaall-trn_lr5e-05_bs15'

mixed_decoder_train_path = TRAIN_ENCDEC_BERT_PATH / mixed_decoder_subdir
mixed_decoder_snapshot_fpath = mixed_decoder_train_path / 'best.pth'
mixed_decoder_model_cfg_fpath = mixed_decoder_train_path / MIXED_DECODER_MODEL_CFG_FNAME

device_name = 'cpu'
# device_name = 'cuda'

device = torch.device(device_name)
print(device)

cpu


## Model loading

In [21]:
model_cfg = parse_yaml_file_as(MixedDecoderCfg, mixed_decoder_model_cfg_fpath)
pprint(model_cfg.dict())

tkz = AutoTokenizer.from_pretrained(model_cfg.enc_bert.pretrained_model_name)
inp_len = model_cfg.enc_bert.inp_len
model_cfg.emb_win_min_size = model_cfg.emb_win_max_size

chkpt = torch.load(mixed_decoder_snapshot_fpath, map_location=device)
model = MixedDecoder(model_cfg, tkz).to(device)
model.load_pretrained(chkpt)
del chkpt
model.eval()
None

{'d_model': 768,
 'decoder_model_name': 'gpt2',
 'decoder_type': <MixedDecoderType.Gpt2: 'gpt2'>,
 'emb_exp_rate': 4,
 'emb_win_max_size': 10,
 'emb_win_min_size': 2,
 'enc_bert': {'d_model': 768,
              'emb2_tok_name': '',
              'emb_type': <BertEmbType.Cls: 'cls'>,
              'inp_len': 128,
              'pad_token_id': 0,
              'pretrained_model_name': 'bert-base-uncased',
              'tokenizer_name': 'bert-base-uncased'},
 'max_seq_len': 384,
 'min_next_toks': 64,
 'prompt_all': False,
 'train_cfg': {'batch_size': 15,
               'freeze_encoder': False,
               'learning_rate': 5e-05,
               'learning_rate_scheduler': {'cls_name': 'ReduceLROnPlateau',
                                           'module_path': 'torch.optim.lr_scheduler',
                                           'params': {'factor': 0.5,
                                                      'min_lr': 1e-08,
                                                      'mode'

## Load QnA datasets (train + val)

In [5]:
max_chunks = max(model_cfg.emb_win_max_size, 1)
qna_agg_train, qna_agg_val = load_qna_datasets(
    tkz=tkz, inp_len=inp_len, max_chunks=max_chunks,
    cache_dir=str(DATA_PATH),
)
print(f'Train size: {len(qna_agg_train)}. Val size: {len(qna_agg_val)}.')
print(f'Train datasets: {len(qna_agg_train.datasets)}. Val datasets: {len(qna_agg_val.datasets)}.')
for i, ds in enumerate(qna_agg_train.datasets):
    print(f'  Train ds {i}: {type(ds).__name__} — {len(ds)} items')
for i, ds in enumerate(qna_agg_val.datasets):
    print(f'  Val ds {i}: {type(ds).__name__} — {len(ds)} items')

SQuAD v2 loaded: train=130319, val=11873


Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/235 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/24 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

TriviaQA loaded: train=138384, val=17944
NewsQA loaded: train=74160, val=4212
MRQA loaded (SearchQA + HotpotQA): train=190312, val=22881
AdversarialQA loaded: train=30000, val=3000
QuAC loaded: train=11567 dialogues (83568 turns), val=1000 dialogues (7354 turns)
CoQA loaded: train=7199 dialogues (108647 turns), val=500 dialogues (7983 turns)
Train size: 889314. Val size: 69240.
Train datasets: 8. Val datasets: 8.
  Train ds 0: SquadV2Dataset — 130319 items
  Train ds 1: NaturalQuestionsDataset — 307373 items
  Train ds 2: TriviaQADataset — 138384 items
  Train ds 3: NewsqaDataset — 74160 items
  Train ds 4: MrqaDataset — 190312 items
  Train ds 5: AdversarialqaDataset — 30000 items
  Train ds 6: QuacDataset — 11567 items
  Train ds 7: CoqaDataset — 7199 items
  Val ds 0: SquadV2Dataset — 11873 items
  Val ds 1: NaturalQuestionsDataset — 7830 items
  Val ds 2: TriviaQADataset — 17944 items
  Val ds 3: NewsqaDataset — 4212 items
  Val ds 4: MrqaDataset — 22881 items
  Val ds 5: Adversari

## Load test splits (where available)

In [7]:
from datasets import load_dataset as hf_load_dataset
from mllm.data.qna.ds_03_triviaqa import TriviaQADataset, TRIVIAQA_HF_ID, TRIVIAQA_SUBSET
from mllm.data.qna.ds_07_mrqa import MrqaDataset, MRQA_HF_ID
from mllm.data.qna.ds_08_adversarialqa import AdversarialqaDataset, ADVERSARIALQA_HF_ID, ADVERSARIALQA_SUBSET

ds_kwargs = dict(tkz=tkz, inp_len=inp_len, max_chunks=max_chunks, device=device)

test_datasets: List[QnaBaseDataset] = []

# TriviaQA test split
ds_triviaqa_test_hf = hf_load_dataset(TRIVIAQA_HF_ID, TRIVIAQA_SUBSET, split='test', cache_dir=str(DATA_PATH))
test_datasets.append(TriviaQADataset(ds=ds_triviaqa_test_hf, **ds_kwargs))
print(f'TriviaQA test: {len(test_datasets[-1])} items')

# MRQA test split
ds_mrqa_test_hf = hf_load_dataset(MRQA_HF_ID, split='test', cache_dir=str(DATA_PATH))
test_datasets.append(MrqaDataset(ds=ds_mrqa_test_hf, **ds_kwargs))
print(f'MRQA test: {len(test_datasets[-1])} items')

# AdversarialQA test split
ds_advqa_test_hf = hf_load_dataset(ADVERSARIALQA_HF_ID, ADVERSARIALQA_SUBSET, split='test', cache_dir=str(DATA_PATH))
test_datasets.append(AdversarialqaDataset(ds=ds_advqa_test_hf, **ds_kwargs))
print(f'AdversarialQA test: {len(test_datasets[-1])} items')

qna_agg_test = QnaDatasetAgg(test_datasets, device=device)
print(f'\nTest aggregate size: {len(qna_agg_test)}')

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

TriviaQA test: 17210 items
MRQA test: 9633 items
AdversarialQA test: 3000 items

Test aggregate size: 29843


## Autoregressive QnA generation function

In [8]:
@torch.no_grad()
def generate_qna(
    model: MixedDecoder, qna_batch: QnaBatch,
    max_new_tokens: int = 100, temperature: float = 1.0,
) -> torch.Tensor:
    """Autoregressive generation for QnA: encode context chunks, build per-sample
    context embedding windows, then generate answer tokens one by one.

    Args:
        model: MixedDecoder model in eval mode
        qna_batch: QnaBatch with context chunks, prompts, and answer tokens
        max_new_tokens: maximum number of tokens to generate
        temperature: sampling temperature (<=0 means greedy argmax)

    Returns:
        generated_toks: (batch_size, max_new_tokens) - generated token ids
    """
    cfg = model.cfg
    batch_size = len(qna_batch.ctx_chunk_counts)
    device = qna_batch.ctx_chunks_toks.device

    # 1. Encode all context chunks
    all_enc_embs = model.run_enc(qna_batch.ctx_chunks_toks, qna_batch.ctx_chunks_att_mask)

    # 2. Build per-sample context windows (own chunks + zero-padding if needed)
    emb_win_max = max(cfg.emb_win_max_size, 1)
    target_win_size = emb_win_max

    chunk_offsets = [0]
    for c in qna_batch.ctx_chunk_counts:
        chunk_offsets.append(chunk_offsets[-1] + c)

    ctx_embs_raw = torch.zeros((batch_size, target_win_size, cfg.d_model), device=device)
    for i in range(batch_size):
        start, end = chunk_offsets[i], chunk_offsets[i + 1]
        own = all_enc_embs[start:end]
        n_own = min(own.shape[0], target_win_size)
        ctx_embs_raw[i, :n_own] = own[:n_own]

    # 3. Apply emb_exp expansion or projection
    if cfg.emb_exp_rate > 0:
        exp_embs = model.emb_exp(ctx_embs_raw)
        exp_embs = exp_embs.view(batch_size, target_win_size * cfg.emb_exp_rate, model.d_dec)
        ctx_embs = exp_embs
    else:
        ctx_embs = ctx_embs_raw

    if model.enc_proj is not None:
        ctx_embs = model.enc_proj(ctx_embs)

    n_ctx = ctx_embs.shape[1]
    sep_len = 1 if cfg.use_sep else 0

    # 4. Build per-sample initial prefix: [ctx_embs, (sep), prompt_embs]
    prompt_lens = qna_batch.prompt_lengths
    max_prefix_len = n_ctx + sep_len + max(prompt_lens)

    input_embs = torch.zeros((batch_size, max_prefix_len, model.d_dec), device=device)
    attention_mask = torch.zeros((batch_size, max_prefix_len), dtype=torch.long, device=device)

    prompt_embs_all = model.word_embeddings(qna_batch.prompt_toks)
    if cfg.use_sep:
        sep_emb = model.word_embeddings(
            torch.full((1, 1), model.sep_token_id, dtype=torch.long, device=device)
        )

    for i in range(batch_size):
        pos = 0
        input_embs[i, :n_ctx] = ctx_embs[i]
        attention_mask[i, :n_ctx] = 1
        pos = n_ctx

        if cfg.use_sep:
            input_embs[i, pos:pos + 1] = sep_emb[0]
            attention_mask[i, pos:pos + 1] = 1
            pos += 1

        pl = prompt_lens[i]
        input_embs[i, pos:pos + pl] = prompt_embs_all[i, :pl]
        attention_mask[i, pos:pos + pl] = 1

    # 5. Autoregressive generation
    generated_ids = []
    for step in range(max_new_tokens):
        total_len = input_embs.shape[1]
        pos_ids = torch.arange(total_len, device=device).unsqueeze(0)
        pos_embs = model.pos_emb(pos_ids)
        embs_with_pos = input_embs + pos_embs

        logits = model.run_decoder(embs_with_pos, attention_mask)
        next_logits = logits[:, -1, :]

        if temperature <= 0:
            next_tok = torch.argmax(next_logits, dim=-1)
        else:
            probs = torch.softmax(next_logits / temperature, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1).squeeze(-1)

        generated_ids.append(next_tok)

        next_emb = model.word_embeddings(next_tok.unsqueeze(1))
        input_embs = torch.cat([input_embs, next_emb], dim=1)
        attention_mask = torch.cat([attention_mask, torch.ones((batch_size, 1), dtype=torch.long, device=device)], dim=1)

    return torch.stack(generated_ids, dim=1)

## Helper: run inference on a split

In [ ]:
def run_qna_inference(
    model: MixedDecoder, tkz: PreTrainedTokenizer,
    ds_agg: QnaDatasetAgg, split_name: str,
    batch_off: int = 0, batch_size: int = 5,
    max_new_tokens: int = 60,
):
    """Run teacher-forced forward pass and autoregressive generation on a QnaDatasetAgg split."""
    cfg = model.cfg
    inds = list(range(batch_off, batch_off + batch_size))
    batch = ds_agg.get_batch(inds)

    # --- Teacher-forced forward pass ---
    with torch.no_grad():
        loss_dict, logits = model.run_on_qna(batch)
    print(f'=== {split_name} — teacher-forced (batch_off={batch_off}, batch_size={batch_size}) ===')
    print(f'Loss: {loss_dict}')
    print(f'Logits shape: {logits.shape}')
    print()

    emb_win_max = max(cfg.emb_win_max_size, 1)
    n_ctx = emb_win_max
    if cfg.emb_exp_rate > 0:
        n_ctx = n_ctx * cfg.emb_exp_rate
    sep_len = 1 if cfg.use_sep else 0

    chunk_offset = 0
    for i in range(batch_size):
        n_chunks = batch.ctx_chunk_counts[i]
        pl = batch.prompt_lengths[i]
        al = int(batch.ans_att_mask[i].sum().item())
        target_start = n_ctx + sep_len + pl - 1

        pred_logits = logits[i, target_start:target_start + al, :]
        pred_toks = torch.argmax(pred_logits, dim=-1).cpu().numpy()
        gt_toks = batch.ans_toks[i, :al].cpu().numpy()

        ds_label = ''
        if batch.ds_src is not None:
            ds_label = f' [{ds_agg.ds_name(batch.ds_src[i])}]'
        print(f'--- Sample {i} ({n_chunks} ctx chunks){ds_label} ---')
        for ci in range(n_chunks):
            chunk_toks = batch.ctx_chunks_toks[chunk_offset + ci].cpu().numpy()
            print(f'  ctx chunk {ci}: {tkz.decode(chunk_toks, skip_special_tokens=True)[:200]}...')
        chunk_offset += n_chunks

        prompt_ids = batch.prompt_toks[i, :pl].cpu().numpy()
        print(f'  prompt:     {tkz.decode(prompt_ids)}')
        print(f'  GT answer:  {tkz.decode(gt_toks)}')
        print(f'  TF predict: {tkz.decode(pred_toks)}')
        print()

    # --- Autoregressive generation ---
    gen_toks = generate_qna(
        model, batch,
        max_new_tokens=max_new_tokens, temperature=0,
    )
    gen_np = gen_toks.cpu().numpy()

    print(f'=== {split_name} — autoregressive generation (max_new_tokens={max_new_tokens}) ===')
    print(f'gen_toks shape: {gen_toks.shape}')
    print()

    chunk_offset = 0
    for i in range(batch_size):
        n_chunks = batch.ctx_chunk_counts[i]
        pl = batch.prompt_lengths[i]
        al = int(batch.ans_att_mask[i].sum().item())
        gt_toks = batch.ans_toks[i, :al].cpu().numpy()
        prompt_ids = batch.prompt_toks[i, :pl].cpu().numpy()
        gen_ids = gen_np[i]

        ds_label = ''
        if batch.ds_src is not None:
            ds_label = f' [{ds_agg.ds_name(batch.ds_src[i])}]'
        print(f'--- Sample {i} ({n_chunks} ctx chunks){ds_label} ---')
        for ci in range(n_chunks):
            chunk_toks = batch.ctx_chunks_toks[chunk_offset + ci].cpu().numpy()
            print(f'  ctx chunk {ci}: {tkz.decode(chunk_toks, skip_special_tokens=True)[:200]}...')
        chunk_offset += n_chunks

        print(f'  prompt:    {tkz.decode(prompt_ids)}')
        print(f'  GT answer: {tkz.decode(gt_toks)}')
        print(f'  generated: {tkz.decode(gen_ids)}')
        if batch.answerable is not None:
            print(f'  answerable: {batch.answerable[i]}')
        print()

## Inference on train split

In [22]:
run_qna_inference(model, tkz, qna_agg_train, split_name='TRAIN', batch_off=0, batch_size=5, max_new_tokens=60)

=== TRAIN — teacher-forced (batch_off=0, batch_size=5) ===
Loss: {'loss': tensor(2.3351)}
Logits shape: torch.Size([5, 60, 50257])

--- Sample 0 (2 ctx chunks) ---
  ctx chunk 0: beyonce giselle knowles - carter ( / biːˈjɒnseɪ / bee - yon - say ) ( born september 4, 1981 ) is an american singer, songwriter, record producer and actress. born and raised in houston, texas, she pe...
  ctx chunk 1: ( 2003 ), which established her as a solo artist worldwide, earned five grammy awards and featured the billboard hot 100 number - one singles " crazy in love " and " baby boy "....
  prompt:     question : when did beyonce start becoming popular? answer :
  GT answer:  in the late 1990s [SEP]
  TF predict: no the 1960s 1960s [SEP]

--- Sample 1 (2 ctx chunks) ---
  ctx chunk 0: beyonce giselle knowles - carter ( / biːˈjɒnseɪ / bee - yon - say ) ( born september 4, 1981 ) is an american singer, songwriter, record producer and actress. born and raised in houston, texas, she pe...
  ctx chunk 1: ( 

## Inference on validation split

In [11]:
run_qna_inference(model, tkz, qna_agg_val, split_name='VAL', batch_off=0, batch_size=5, max_new_tokens=60)

=== VAL — teacher-forced (batch_off=0, batch_size=5) ===
Loss: {'loss': tensor(1.9117)}
Logits shape: torch.Size([5, 58, 30522])

--- Sample 0 (2 ctx chunks) ---
  ctx chunk 0: the normans ( norman : nourmands ; french : normands ; latin : normanni ) were the people who in the 10th and 11th centuries gave their name to normandy, a region in france. they were descended from n...
  ctx chunk 1: distinct cultural and ethnic identity of the normans emerged initially in the first half of the 10th century, and it continued to evolve over the succeeding centuries....
  prompt:     question : in what country is normandy located? answer :
  GT answer:  france [SEP]
  TF predict: denmark [SEP]

--- Sample 1 (2 ctx chunks) ---
  ctx chunk 0: the normans ( norman : nourmands ; french : normands ; latin : normanni ) were the people who in the 10th and 11th centuries gave their name to normandy, a region in france. they were descended from n...
  ctx chunk 1: distinct cultural and ethnic identity of

## Inference on test split

In [12]:
run_qna_inference(model, tkz, qna_agg_test, split_name='TEST', batch_off=0, batch_size=5, max_new_tokens=60)

Token indices sequence length is longer than the specified maximum sequence length for this model (3174 > 512). Running this sequence through the model will result in indexing errors


=== TEST — teacher-forced (batch_off=0, batch_size=5) ===
Loss: {'loss': tensor(7.8070)}
Logits shape: torch.Size([5, 64, 30522])

--- Sample 0 (10 ctx chunks) ---
  ctx chunk 0: airports in eritrea, eritrea airports map massawa international airport flights to eritrea flights to eritrea are managed by the many national and international airlines that function from the various...
  ctx chunk 1: from these cities to eritrea and also from eritrea to these cities. a major problem with the aviation industry in africa is that not all - important african cities are interconnected by airways. hence...
  ctx chunk 2: this does not mean compromising on the quality of service. eritrea airline eritrea airlines is among the chief airlines that are functional in eritrea, the other two being the saudi arabian airlines a...
  ctx chunk 3: airlines, serving in the african continent as well as those outside. the service provided by the eritrea airlines when east african air safari was banned is an exam

## More examples from different offsets

In [13]:
run_qna_inference(model, tkz, qna_agg_train, split_name='TRAIN', batch_off=100, batch_size=5, max_new_tokens=60)

=== TRAIN — teacher-forced (batch_off=100, batch_size=5) ===
Loss: {'loss': tensor(2.2019)}
Logits shape: torch.Size([5, 66, 30522])

--- Sample 0 (2 ctx chunks) ---
  ctx chunk 0: the remaining band members recorded " independent women part i ", which appeared on the soundtrack to the 2000 film, charlie ' s angels. it became their best - charting single, topping the u. s. billb...
  ctx chunk 1: was released in may 2001, luckett and roberson filed a lawsuit claiming that the songs were aimed at them. the album debuted at number one on the u. s. billboard 200, with first - week sales of 663, 0...
  prompt:     question : how many weeks did their single " independent women part i " stay on top? answer :
  GT answer:  eleven [SEP]
  TF predict: three [SEP]

--- Sample 1 (2 ctx chunks) ---
  ctx chunk 0: the remaining band members recorded " independent women part i ", which appeared on the soundtrack to the 2000 film, charlie ' s angels. it became their best - charting single, topping th

In [14]:
run_qna_inference(model, tkz, qna_agg_val, split_name='VAL', batch_off=100, batch_size=5, max_new_tokens=60)

=== VAL — teacher-forced (batch_off=100, batch_size=5) ===
Loss: {'loss': tensor(0.7801)}
Logits shape: torch.Size([5, 60, 30522])

--- Sample 0 (1 ctx chunks) ---
  ctx chunk 0: in 1066, duke william ii of normandy conquered england killing king harold ii at the battle of hastings. the invading normans and their descendants replaced the anglo - saxons as the ruling class of e...
  prompt:     question : what battle took place in the 10th century? answer :
  GT answer:  noanswer [SEP]
  TF predict: noanswer [SEP]

--- Sample 1 (1 ctx chunks) ---
  ctx chunk 0: in 1066, duke william ii of normandy conquered england killing king harold ii at the battle of hastings. the invading normans and their descendants replaced the anglo - saxons as the ruling class of e...
  prompt:     question : who replaced the normans as the ruling class? answer :
  GT answer:  noanswer [SEP]
  TF predict: noanswer [SEP]

--- Sample 2 (1 ctx chunks) ---
  ctx chunk 0: in 1066, duke william ii of normandy conque

In [15]:
run_qna_inference(model, tkz, qna_agg_test, split_name='TEST', batch_off=100, batch_size=5, max_new_tokens=60)

=== TEST — teacher-forced (batch_off=100, batch_size=5) ===
Loss: {'loss': tensor(7.5894)}
Logits shape: torch.Size([5, 65, 30522])

--- Sample 0 (10 ctx chunks) ---
  ctx chunk 0: hair blu - ray movie > hair bray for sale product description hair movie was released jun 07, 2011 by the mgm ( lasers ) studio. in what is widely considered to be even better than the broadway stage ...
  ctx chunk 1: is adopted by a group of flower children, led by berger ( treat williams ) and including jeannie ( annie golden ), who take him on a series of counter - cultural adventures that introduce claude to ha...
  ctx chunk 2: option on select cd ' s displaying the prerip icon. this option allows you to download the mp3 version of that cd immediately after your purchase. the physical cd will still be shipped to you. if you ...
  ctx chunk 3: ##rip is only available to customers in the united states. this is a limitation placed on us by the record labels. hair movie review & film summary ( 1979 ) | rog

## Arbitrary context + question inference

In [16]:
def build_qna_batch_from_text(
    context: str, question: str, answer: str,
    tkz: PreTrainedTokenizer, inp_len: int, max_chunks: int,
    max_ans_toks: int = 100, max_prompt_toks: int = 100,
    device: Optional[torch.device] = None,
) -> QnaBatch:
    """Build a single-sample QnaBatch from raw context, question, and answer strings."""
    if device is None:
        device = torch.device('cpu')
    pad_id = tkz.pad_token_id
    cls_id = tkz.cls_token_id
    sep_id = tkz.sep_token_id

    # Chunk context
    ctx_toks = tkz(context, add_special_tokens=False).input_ids
    chunk_content_len = inp_len - 2
    chunks = []
    for start in range(0, len(ctx_toks), chunk_content_len):
        content = ctx_toks[start:start + chunk_content_len]
        chunk = [cls_id] + content + [sep_id]
        chunks.append(chunk)
        if len(chunks) >= max_chunks:
            break

    # Tokenize prompt: "Question: {q} Answer:"
    prompt_prefix = tkz('Question: ', add_special_tokens=False).input_ids
    prompt_suffix = tkz(' Answer:', add_special_tokens=False).input_ids
    q_toks = tkz(question, add_special_tokens=False).input_ids
    budget = max_prompt_toks - len(prompt_prefix) - len(prompt_suffix)
    if len(q_toks) > budget:
        q_toks = q_toks[-budget:]
    prompt_toks = prompt_prefix + q_toks + prompt_suffix

    # Tokenize answer
    ans_toks = tkz(answer, add_special_tokens=True).input_ids[1:]  # strip leading CLS
    if len(ans_toks) > max_ans_toks:
        ans_toks = ans_toks[:max_ans_toks]

    # Build tensors
    n_chunks = len(chunks)
    ctx_t = torch.full((n_chunks, inp_len), pad_id, dtype=torch.long, device=device)
    ctx_att = torch.zeros((n_chunks, inp_len), dtype=torch.long, device=device)
    for i, ch in enumerate(chunks):
        n = min(len(ch), inp_len)
        ctx_t[i, :n] = torch.tensor(ch[:n], dtype=torch.long, device=device)
        ctx_att[i, :n] = 1

    prompt_t = torch.tensor([prompt_toks], dtype=torch.long, device=device)
    prompt_att = torch.ones((1, len(prompt_toks)), dtype=torch.long, device=device)

    ans_t = torch.tensor([ans_toks], dtype=torch.long, device=device)
    ans_att = torch.ones((1, len(ans_toks)), dtype=torch.long, device=device)

    return QnaBatch(
        ctx_chunks_toks=ctx_t,
        ctx_chunks_att_mask=ctx_att,
        ctx_chunk_counts=[n_chunks],
        prompt_toks=prompt_t,
        prompt_att_mask=prompt_att,
        prompt_lengths=[len(prompt_toks)],
        ans_toks=ans_t,
        ans_att_mask=ans_att,
        answerable=[True],
    )


def run_arbitrary_qna(
    model: MixedDecoder, tkz: PreTrainedTokenizer,
    context: str, question: str, answer: str = '',
    max_new_tokens: int = 60, temperature: float = 0,
):
    """Run inference on arbitrary context/question/answer text."""
    cfg = model.cfg
    max_chunks = max(cfg.emb_win_max_size, 1)
    inp_len = cfg.enc_bert.inp_len

    batch = build_qna_batch_from_text(
        context, question, answer, tkz, inp_len, max_chunks,
        device=next(model.parameters()).device,
    )

    # Teacher-forced forward pass (if answer is provided)
    if answer:
        with torch.no_grad():
            loss_dict, logits = model.run_on_qna(batch)

        emb_win_max = max(cfg.emb_win_max_size, 1)
        n_ctx = emb_win_max
        if cfg.emb_exp_rate > 0:
            n_ctx *= cfg.emb_exp_rate
        sep_len = 1 if cfg.use_sep else 0

        pl = batch.prompt_lengths[0]
        al = int(batch.ans_att_mask[0].sum().item())
        target_start = n_ctx + sep_len + pl - 1

        pred_logits = logits[0, target_start:target_start + al, :]
        pred_toks = torch.argmax(pred_logits, dim=-1).cpu().numpy()
        gt_toks = batch.ans_toks[0, :al].cpu().numpy()

        print(f'Loss: {loss_dict}')
        print(f'GT answer:  {tkz.decode(gt_toks)}')
        print(f'TF predict: {tkz.decode(pred_toks)}')
        print()

    # Autoregressive generation
    gen_toks = generate_qna(model, batch, max_new_tokens=max_new_tokens, temperature=temperature)
    gen_np = gen_toks[0].cpu().numpy()

    print(f'Context ({batch.ctx_chunk_counts[0]} chunks): {context[:300]}{"..." if len(context) > 300 else ""}')
    print(f'Question: {question}')
    if answer:
        print(f'Answer (GT): {answer}')
    print(f'Generated:   {tkz.decode(gen_np)}')

In [17]:
context_1 = (
    "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. "
    "It is named after the engineer Gustave Eiffel, whose company designed and built the tower "
    "from 1887 to 1889 as the centerpiece of the 1889 World's Fair. Although initially criticised "
    "by some of France's leading artists and intellectuals for its design, it has since become a "
    "global cultural icon of France and one of the most recognisable structures in the world. "
    "The tower is 330 metres (1,083 ft) tall, about the same height as an 81-storey building, "
    "and the tallest structure in Paris."
)
run_arbitrary_qna(model, tkz, context_1, question="How tall is the Eiffel Tower?", answer="330 metres")

Loss: {'loss': tensor(6.3000)}
GT answer:  330 metres [SEP]
TF predict: no [SEP] [SEP]

Context (2 chunks): The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel, whose company designed and built the tower from 1887 to 1889 as the centerpiece of the 1889 World's Fair. Although initially criticised by some of France's leading a...
Question: How tall is the Eiffel Tower?
Answer (GT): 330 metres
Generated:   one hundred and eighty - two [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] anniversary [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] anniversary [SEP]


In [18]:
context_2 = (
    "Photosynthesis is a process used by plants and other organisms to convert light energy into "
    "chemical energy that, through cellular respiration, can later be released to fuel the organism's "
    "activities. In most cases, oxygen is also released as a waste product that sustains nearly all "
    "life on Earth. Most plants, algae, and cyanobacteria perform photosynthesis; such organisms are "
    "called photoautotrophs. Photosynthesis is largely responsible for producing and maintaining the "
    "oxygen content of the Earth's atmosphere, and supplies most of the biological energy necessary "
    "for complex life on Earth."
)
run_arbitrary_qna(model, tkz, context_2, question="What is photosynthesis?", answer="a process used by plants and other organisms to convert light energy into chemical energy")

Loss: {'loss': tensor(1.7764)}
GT answer:  a process used by plants and other organisms to convert light energy into chemical energy [SEP]
TF predict: no fire [SEP] on humans and other chemicals to transform light energy into nuclear energy [SEP]

Context (1 chunks): Photosynthesis is a process used by plants and other organisms to convert light energy into chemical energy that, through cellular respiration, can later be released to fuel the organism's activities. In most cases, oxygen is also released as a waste product that sustains nearly all life on Earth. M...
Question: What is photosynthesis?
Answer (GT): a process used by plants and other organisms to convert light energy into chemical energy
Generated:   photosynthesis [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] fission fission fission fission fission fission fission fission fission fission fission f

In [19]:
# Generation-only (no ground-truth answer)
context_3 = (
    "The Apollo 11 mission was the spaceflight that first landed humans on the Moon. "
    "Commander Neil Armstrong and lunar module pilot Buzz Aldrin landed the Apollo Lunar Module "
    "Eagle on July 20, 1969. Armstrong became the first person to step onto the lunar surface "
    "six hours and 39 minutes later, and Aldrin joined him 19 minutes after that. They spent "
    "about two and a quarter hours together exploring the site they had named Tranquility Base "
    "upon landing. Michael Collins flew the command module Columbia alone in lunar orbit while "
    "they were on the Moon's surface."
)
run_arbitrary_qna(model, tkz, context_3, question="Who was the first person to walk on the Moon?")

Context (1 chunks): The Apollo 11 mission was the spaceflight that first landed humans on the Moon. Commander Neil Armstrong and lunar module pilot Buzz Aldrin landed the Apollo Lunar Module Eagle on July 20, 1969. Armstrong became the first person to step onto the lunar surface six hours and 39 minutes later, and Aldr...
Question: Who was the first person to walk on the Moon?
Generated:   john herschel [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] explosion [SEP] deck [SEP] [SEP] explosion explosion explosion explosion explosion explosion explosion explosion explosion explosion explosion explosion explosion explosion explosion explosion explosion explosion explosion
